# Notebook 05 — Model Comparison
Final side-by-side comparison of all candidates. Latency benchmarks. Save full report for `docs/model_selection.md`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
from pathlib import Path

from evaluation.compare_models import compare_l2a, compare_l2b, save_full_report
from evaluation.benchmark import benchmark_all, print_report

EXPORTS = Path("../exported_models")

## 1. Load saved results

In [ ]:
with open(EXPORTS / "l2a_results.json") as f:
    l2a_data = json.load(f)
with open(EXPORTS / "l2b_results.json") as f:
    l2b_data = json.load(f)

l2a_results = l2a_data["results"]
l2b_results = l2b_data["results"]
l2a_winner  = l2a_data["winner"]
l2b_winner  = l2b_data["winner"]

print(f"L2A winner: {l2a_winner}")
print(f"L2B winner: {l2b_winner}")

## 2. Layer 2A comparison table

In [ ]:
df_l2a = compare_l2a(l2a_results)
df_l2a

## 3. Layer 2B comparison table

In [ ]:
df_l2b = compare_l2b(l2b_results)
df_l2b

## 4. ONNX latency benchmark — all exported models

In [ ]:
lat_results = benchmark_all(str(EXPORTS))
print_report(lat_results)

## 5. Summary chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# L2A: FPR vs TPR
models_l2a = [r["model"] for r in l2a_results]
fprs = [r["fpr"] for r in l2a_results]
tprs = [r["tpr"] for r in l2a_results]
colors = ["steelblue" if m == l2a_winner else "lightgray" for m in models_l2a]
bars = axes[0].bar(models_l2a, fprs, color=colors, alpha=0.8, label="FPR (lower=better)")
axes[0].bar(models_l2a, [-t for t in tprs], color=["coral" if m == l2a_winner else "lightsalmon"
            for m in models_l2a], alpha=0.6, label="TPR (higher=better, shown negative)")
axes[0].set_title("Layer 2A — FPR vs TPR")
axes[0].axhline(0, color="black", lw=0.5)
axes[0].legend(fontsize=8)
axes[0].set_ylabel("Rate")

# L2B: Macro F1
models_l2b = [r["model"] for r in l2b_results]
f1s = [r["macro_f1"] for r in l2b_results]
colors_b = ["steelblue" if m == l2b_winner else "lightgray" for m in models_l2b]
axes[1].bar(models_l2b, f1s, color=colors_b)
axes[1].set_ylim(0.8, 1.0)
axes[1].set_title("Layer 2B — Macro F1")
axes[1].set_ylabel("Macro F1")
for bar, f1 in zip(axes[1].patches, f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f"{f1:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("../data/processed/05_model_comparison.png", dpi=120)
plt.show()

## 6. Save full report

In [ ]:
save_full_report(l2a_results, l2b_results, lat_results,
                 str(EXPORTS / "full_model_report.json"))
print("Report saved. Copy results to docs/model_selection.md")

## 7. vs Base Paper benchmarks

In [ ]:
print("=== Comparison vs Base Papers ===")
print()
print("Base Paper 1 (ADL-WAF, Sameh & Selim):")
print("  Detection accuracy: 99.88%")
print("  Precision:          100%")
print()
print("Base Paper 2 (LSTM+GRU ensemble, Babaey & Faragardi 2025):")
print("  Accuracy:    97.58%")
print("  Precision:   99.99%")
print("  Recall:      97.52%")
print("  Specificity: 99.76%")
print("  FPR:         0.2%")
print()
print("Our Layer 2A results:")
for r in l2a_results:
    print(f"  {r['model']}: TPR={r['tpr']:.4f}  FPR={r['fpr']:.4f}  AUC={r['auc']:.4f}")
print()
print("Our Layer 2B results:")
for r in l2b_results:
    print(f"  {r['model']}: macro_F1={r['macro_f1']:.4f}  acc={r['accuracy']:.4f}")